# 🤟 Sign-to-Speech — Pipeline complet
### CNN (MediaPipe) → RNN (LSTM) → Prédiction en temps réel

**Ordre d'exécution :** Lancer les cellules **de haut en bas**, dans l'ordre.

---

## ÉTAPE 1 — Installation des dépendances

In [1]:
# Installation de MediaPipe
!pip install mediapipe -q

# Téléchargement du modèle CNN (Hand Landmarker de Google)
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

print('✅ Installation terminée !')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.6 MB/s eta 0:00:00
✅ Installation terminée !


## ÉTAPE 2 — Imports et initialisation du CNN (MediaPipe)

In [2]:
import os
import cv2
import numpy as np
import mediapipe as mp
import base64
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

# Initialisation du CNN MediaPipe (détecteur de 21 points de la main)
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

print('✅ CNN MediaPipe initialisé et prêt !')

✅ CNN MediaPipe initialisé et prêt !


## ÉTAPE 3 — Création de la structure de dossiers

> 💡 **Ajoute tes signes** dans le tableau `signes` ci-dessous.
> Pour commencer, on travaille avec `'bonjour'` et `'merci'`.

In [5]:
# ➡️ MODIFIE CETTE LISTE pour ajouter / retirer des signes
signes = ['bonjour', 'neutre']

DATA_PATH = 'DATA_SIGNES'

for signe in signes:
    path = os.path.join(DATA_PATH, signe)
    os.makedirs(path, exist_ok=True)
    print(f'📁 Dossier créé : {path}')

print('\n✅ Structure de dossiers prête !')

📁 Dossier créé : DATA_SIGNES/bonjour
📁 Dossier créé : DATA_SIGNES/neutre

✅ Structure de dossiers prête !


## ÉTAPE 4 — Enregistrement vidéo

> 💡 Lance cette cellule autant de fois que tu veux.
> Change `NOM_DU_SIGNE` et `NOM_FICHIER` à chaque fois.
> **Objectif : 6 à 10 vidéos par signe.**

In [6]:
from google.colab import files

# ➡️ MODIFIE ce nom
NOM_DU_SIGNE = 'neutre'

dossier_cible = os.path.join(DATA_PATH, NOM_DU_SIGNE)
os.makedirs(dossier_cible, exist_ok=True)

print(f'📁 Sélectionne tes vidéos pour le signe : {NOM_DU_SIGNE}')
uploaded = files.upload()

for i, (nom_fichier, contenu) in enumerate(uploaded.items()):
    extension = os.path.splitext(nom_fichier)[1]
    chemin = os.path.join(dossier_cible, f'video_{i+1:02d}{extension}')
    with open(chemin, 'wb') as f:
        f.write(contenu)
    print(f'✅ Sauvegardé : {chemin}')

print(f'\n📊 {len(uploaded)} vidéo(s) uploadée(s) pour "{NOM_DU_SIGNE}"')

📁 Sélectionne tes vidéos pour le signe : neutre


Saving 20260512_213526000_iOS.MOV to 20260512_213526000_iOS.MOV
Saving 20260512_213531000_iOS.MOV to 20260512_213531000_iOS.MOV
Saving 20260512_213536000_iOS.MOV to 20260512_213536000_iOS.MOV
Saving 20260512_213541000_iOS.MOV to 20260512_213541000_iOS.MOV
Saving 20260512_213545000_iOS.MOV to 20260512_213545000_iOS.MOV
Saving 20260512_213550000_iOS.MOV to 20260512_213550000_iOS.MOV
Saving 20260512_213555000_iOS.MOV to 20260512_213555000_iOS.MOV
Saving 20260512_213600000_iOS.MOV to 20260512_213600000_iOS.MOV
Saving 20260512_213625000_iOS.MOV to 20260512_213625000_iOS.MOV
Saving 20260512_213630000_iOS.MOV to 20260512_213630000_iOS.MOV
✅ Sauvegardé : DATA_SIGNES/neutre/video_01.MOV
✅ Sauvegardé : DATA_SIGNES/neutre/video_02.MOV
✅ Sauvegardé : DATA_SIGNES/neutre/video_03.MOV
✅ Sauvegardé : DATA_SIGNES/neutre/video_04.MOV
✅ Sauvegardé : DATA_SIGNES/neutre/video_05.MOV
✅ Sauvegardé : DATA_SIGNES/neutre/video_06.MOV
✅ Sauvegardé : DATA_SIGNES/neutre/video_07.MOV
✅ Sauvegardé : DATA_SIGNES/neut

## ÉTAPE 5 — Extraction des points (CNN → coordonnées)

> Cette cellule parcourt toutes tes vidéos et les transforme en fichiers `.npy`
> (des tableaux de coordonnées). C'est la sortie du CNN.

In [7]:
def extraire_points_video(chemin_video, nom_signe, num_sequence):
    """Lit une vidéo, extrait les 21 landmarks de la main frame par frame,
    et sauvegarde le résultat dans DATA_SIGNES/<signe>/<num>.npy"""
    cap = cv2.VideoCapture(chemin_video)
    sequence_data = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
        detection_result = detector.detect(mp_image)

        if detection_result.hand_landmarks:
            frame_points = []
            for lm in detection_result.hand_landmarks[0]:
                frame_points.extend([lm.x, lm.y, lm.z])
            sequence_data.append(frame_points)
        else:
            # Aucune main détectée → on met des zéros pour garder la continuité
            sequence_data.append([0] * 63)

    cap.release()

    sauvegarde_path = os.path.join(DATA_PATH, nom_signe, f'{num_sequence}.npy')
    np.save(sauvegarde_path, sequence_data)
    print(f'  ✅ {chemin_video} → {sauvegarde_path} ({len(sequence_data)} frames)')


# Lancement de l'extraction pour tous les signes
print('🔍 Début de l\'extraction des points...\n')

for signe in signes:
    dossier = os.path.join(DATA_PATH, signe)
    fichiers_video = [f for f in os.listdir(dossier) if f.lower().endswith(('.mp4', '.mov', '.webm'))]

    if len(fichiers_video) == 0:
        print(f'⚠️  [{signe}] Aucune vidéo trouvée dans {dossier}')
        continue

    print(f'📂 [{signe}] {len(fichiers_video)} vidéo(s) trouvée(s)')
    for i, nom_fichier in enumerate(sorted(fichiers_video)):
        extraire_points_video(os.path.join(dossier, nom_fichier), signe, i)

print('\n--- EXTRACTION TERMINÉE ---')

🔍 Début de l'extraction des points...

📂 [bonjour] 11 vidéo(s) trouvée(s)
  ✅ DATA_SIGNES/bonjour/video_01.MOV → DATA_SIGNES/bonjour/0.npy (131 frames)
  ✅ DATA_SIGNES/bonjour/video_02.MOV → DATA_SIGNES/bonjour/1.npy (120 frames)
  ✅ DATA_SIGNES/bonjour/video_03.MOV → DATA_SIGNES/bonjour/2.npy (114 frames)
  ✅ DATA_SIGNES/bonjour/video_04.MOV → DATA_SIGNES/bonjour/3.npy (167 frames)
  ✅ DATA_SIGNES/bonjour/video_05.MOV → DATA_SIGNES/bonjour/4.npy (99 frames)
  ✅ DATA_SIGNES/bonjour/video_06.MOV → DATA_SIGNES/bonjour/5.npy (108 frames)
  ✅ DATA_SIGNES/bonjour/video_07.MOV → DATA_SIGNES/bonjour/6.npy (112 frames)
  ✅ DATA_SIGNES/bonjour/video_08.MOV → DATA_SIGNES/bonjour/7.npy (111 frames)
  ✅ DATA_SIGNES/bonjour/video_09.MOV → DATA_SIGNES/bonjour/8.npy (114 frames)
  ✅ DATA_SIGNES/bonjour/video_10.MOV → DATA_SIGNES/bonjour/9.npy (114 frames)
  ✅ DATA_SIGNES/bonjour/video_11.MOV → DATA_SIGNES/bonjour/10.npy (122 frames)
📂 [neutre] 10 vidéo(s) trouvée(s)
  ✅ DATA_SIGNES/neutre/video_01.MO

## ÉTAPE 6 — Chargement des données + préparation pour le RNN

In [8]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

NB_FRAMES = 30  # Nombre de frames par séquence (normalisé)
NB_COORDS = 63  # 21 points × 3 coordonnées (x, y, z)

actions   = np.array(signes)
label_map = {label: num for num, label in enumerate(actions)}

sequences, labels = [], []

for action in actions:
    path = os.path.join(DATA_PATH, action)
    fichiers_npy = [f for f in os.listdir(path) if f.endswith('.npy')]

    print(f'📂 [{action}] {len(fichiers_npy)} séquence(s) chargée(s)')

    for f in fichiers_npy:
        res = np.load(os.path.join(path, f))

        # Normalisation à exactement NB_FRAMES frames
        if len(res) >= NB_FRAMES:
            res = res[:NB_FRAMES]
        else:
            padding = np.zeros((NB_FRAMES - len(res), NB_COORDS))
            res = np.concatenate([res, padding])

        sequences.append(res)
        labels.append(label_map[action])

if len(sequences) == 0:
    print('\n❌ Aucune donnée chargée ! Lance d\'abord les étapes 4 et 5.')
else:
    X = np.array(sequences)
    y = to_categorical(labels).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

    print(f'\n✅ Données prêtes !')
    print(f'   {X.shape[0]} exemples au total')
    print(f'   Forme : {X.shape[1]} frames × {X.shape[2]} coordonnées')
    print(f'   Train : {len(X_train)} | Test : {len(X_test)}')

📂 [bonjour] 11 séquence(s) chargée(s)
📂 [neutre] 10 séquence(s) chargée(s)

✅ Données prêtes !
   21 exemples au total
   Forme : 30 frames × 63 coordonnées
   Train : 18 | Test : 3


## ÉTAPE 7 — Construction et entraînement du RNN (LSTM)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Architecture du réseau
model = Sequential([
    LSTM(64, return_sequences=True, activation='relu', input_shape=(NB_FRAMES, NB_COORDS)),
    LSTM(128, return_sequences=False, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(len(actions), activation='softmax')  # Autant de neurones que de signes
])

model.compile(
    optimizer='Adam',
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)

model.summary()

# EarlyStopping : arrête l'entraînement si ça ne s'améliore plus (évite le surapprentissage)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

print('\n🚀 Début de l\'entraînement...')
history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)
print('\n🧠 Entraînement terminé !')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 141,986 (554.63 KB)

 Trainable params: 141,986 (554.63 KB)

 Non-trainable params: 0 (0.00 B)


🚀 Début de l'entraînement...
Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - categorical_accuracy: 0.5000 - loss: 0.6836 - val_categorical_accuracy: 0.3333 - val_loss: 0.6501
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 648ms/step - categorical_accuracy: 0.5556 - loss: 0.6232 - val_categorical_accuracy: 0.3333 - val_loss: 0.6113
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - categorical_accuracy: 0.5556 - loss: 0.5617 - val_categorical_accuracy: 0.3333 - val_loss: 0.5616
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - categorical_accuracy: 0.5556 - loss: 0.4851 - val_categorical_accuracy: 0.3333 - val_loss: 0.5037
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - categorical_accuracy: 0.5556 - loss: 0.3940 - val_categorical_accuracy: 0.3333 - val_loss: 0.4709
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - categorical_accuracy: 0.5556 - loss: 0.3309 - val_categorical_accuracy: 0.3333 - val_loss: 0.4679
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - categorical_accur

In [10]:
model.save('model_signes.h5')
from google.colab import files
files.download('model_signes.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ÉTAPE 8 — Évaluation du modèle

In [11]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'📊 Précision sur les données de test : {accuracy * 100:.1f}%')

# Sauvegarde du modèle
model.save('model_signes.h5')
print('💾 Modèle sauvegardé : model_signes.h5')

📊 Précision sur les données de test : 100.0%
💾 Modèle sauvegardé : model_signes.h5


## ÉTAPE 9 — Démonstration en temps réel (webcam)

> Lance cette cellule, autorise l'accès à ta webcam, et fais tes signes !
> Le modèle affichera sa prédiction avec son niveau de confiance.

In [4]:
import google.colab.output

# Buffer global pour stocker les frames en cours d'analyse
sequence_buffer = []

def detect_sign(imgData):
    """Reçoit une frame depuis JavaScript, extrait les points,
    et renvoie la prédiction du LSTM quand le buffer est plein."""
    global sequence_buffer

    # Décodage de l'image reçue depuis le navigateur
    header, encoded = imgData.split(',', 1)
    data = base64.b64decode(encoded)
    np_img = np.frombuffer(data, dtype=np.uint8)
    frame = cv2.imdecode(np_img, cv2.IMREAD_COLOR)

    # Extraction CNN (MediaPipe)
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    detection_result = detector.detect(mp_image)

    if detection_result.hand_landmarks:
        res = []
        for lm in detection_result.hand_landmarks[0]:
            res.extend([lm.x, lm.y, lm.z])
        sequence_buffer.append(res)
    else:
        sequence_buffer.append([0] * 63)

    # Prédiction RNN quand on a assez de frames
    if len(sequence_buffer) >= NB_FRAMES:
        input_data = np.expand_dims(sequence_buffer[-NB_FRAMES:], axis=0)
        prediction = model.predict(input_data, verbose=0)

        index     = np.argmax(prediction)
        confiance = prediction[0][index]

        # On garde les 15 dernières frames pour le prochain cycle
        sequence_buffer = sequence_buffer[-25:]

        if confiance > 0.8:  # Seuil de confiance (modifiable)
            return f'{actions[index].upper()} ({int(confiance * 100)}%)'

    return 'Analyse...'


google.colab.output.register_callback('notebook.detect_sign', detect_sign)


def run_live_ia():
    js = Javascript('''
    async function showVideo() {
        const div = document.createElement('div');
        div.style.position = 'relative';
        div.style.display = 'inline-block';

        const video = document.createElement('video');
        video.style.display = 'block';
        video.style.transform = 'scaleX(-1)';
        video.style.borderRadius = '12px';

        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;

        let overlay = document.createElement('div');
        overlay.style.cssText = 'position:absolute;bottom:16px;left:50%;transform:translateX(-50%);' +
                                'color:white;font-size:36px;font-weight:bold;' +
                                'background:rgba(0,0,0,0.5);padding:8px 20px;border-radius:8px;';
        overlay.innerText = 'Prêt...';
        div.appendChild(overlay);

        while (true) {
            canvas.getContext('2d').drawImage(video, 0, 0);
            const imgData = canvas.toDataURL('image/jpeg', 0.7);
            const result = await google.colab.kernel.invokeFunction(
                'notebook.detect_sign', [imgData], {}
            );
            overlay.innerText = result.data['text/plain'].replace(/'/g, '');
        }
    }
    showVideo();
    ''')
    display(js)


print('📷 Lancement de la webcam...')
run_live_ia()

📷 Lancement de la webcam...


<IPython.core.display.Javascript object>